# Vue2 + Vue3 框架 · 第 12 天

## Pinia 状态管理

> 本篇依据第 12 天 Pinia MD 笔记、PPT 和配套代码整理。Pinia 是 Vue 3 官方推荐的状态管理库，API 更简单，适合替代 Vuex。

## 一、Pinia 解决什么问题

组件之间共享的数据可以放到 store 中统一管理。Pinia 的 store 通常包含：

- **state**：原始状态。
- **getters**：基于 state 的派生值，类似 computed。
- **actions**：修改状态和异步业务逻辑。

相比 Vuex，Pinia 不需要 mutations，支持 TypeScript 更自然，store 可以按业务拆分，devtools 也可以直接调试。

## 二、安装和挂载 Pinia

```bash
npm install pinia
```

```js
// main.js
import { createApp } from 'vue'
import { createPinia } from 'pinia'
import App from './App.vue'

const app = createApp(App)
app.use(createPinia())
app.mount('#app')
```

在实际项目中，create-vue 可以在创建阶段直接选择 Pinia；手动安装适合学习和已有项目接入。

## 三、定义 setup store

```js
// stores/counter.js
import { computed, ref } from 'vue'
import { defineStore } from 'pinia'

export const useCounterStore = defineStore('counter', () => {
  const count = ref(0)
  const doubleCount = computed(() => count.value * 2)
  const increment = () => { count.value++ }

  return { count, doubleCount, increment }
})
```

组件中使用：

```vue
<script setup>
import { useCounterStore } from '@/stores/counter'
const counter = useCounterStore()
</script>
<template>
  <p>{{ counter.count }} / {{ counter.doubleCount }}</p>
  <button @click="counter.increment">+1</button>
</template>
```

store id 必须唯一，命名建议使用业务名，如 `user`、`cart`、`channel`。

## 四、state 与 getters

setup store 中用 `ref` 定义 state，用 `computed` 定义 getter：

```js
export const useChannelStore = defineStore('channel', () => {
  const channelList = ref([])
  const activeChannel = ref(null)
  const channelCount = computed(() => channelList.value.length)
  const currentName = computed(() => activeChannel.value?.name || '未选择')

  return { channelList, activeChannel, channelCount, currentName }
})
```

组件直接访问 `store.channelList` 和 `store.channelCount`。getter 是只读派生值，不要在模板中重复写复杂计算。

## 五、actions 与异步请求

Pinia 的 action 可以直接写异步函数：

```js
import { ref } from 'vue'
import { defineStore } from 'pinia'
import axios from 'axios'

export const useChannelStore = defineStore('channel', () => {
  const channelList = ref([])
  const loading = ref(false)

  const getChannelList = async () => {
    loading.value = true
    try {
      const res = await axios.get('http://geek.itheima.net/v1_0/channels')
      channelList.value = res.data.data.channels
    } finally {
      loading.value = false
    }
  }

  return { channelList, loading, getChannelList }
})
```

组件中调用 `store.getChannelList()`。请求、错误和 loading 状态都可以封装在 store 中，多个页面共享同一份结果。

## 六、`storeToRefs` 保持响应式解构

直接解构 store 会失去响应式：

```js
const { count } = store // 不推荐
```

使用 `storeToRefs` 解构 state 和 getter：

```js
import { storeToRefs } from 'pinia'
const store = useChannelStore()
const { channelList, loading, channelCount } = storeToRefs(store)
const { getChannelList } = store // action 直接解构即可
```

模板中可直接使用 `channelList`、`loading`。记住：数据和 getter 用 `storeToRefs`，方法不需要。

## 七、Pinia 调试与持久化

Vue Devtools 可以直接查看 Pinia 的 store、状态变化和 action 调用。开发时给 store 使用清晰的 id，方便定位。

使用持久化插件：

```bash
npm install pinia-plugin-persistedstate
```

```js
// main.js
import { createPinia } from 'pinia'
import persist from 'pinia-plugin-persistedstate'

const pinia = createPinia().use(persist)
app.use(pinia)
```

```js
export const useUserStore = defineStore('user', () => {
  const token = ref('')
  const setToken = value => { token.value = value }
  return { token, setToken }
}, {
  persist: true
})
```

持久化的内容要谨慎选择：token、用户偏好和购物车可以保存，临时 loading、错误对象和大体积列表不宜盲目持久化。

## 八、Pinia 与 Vuex 对照

| Vuex | Pinia |
| --- | --- |
| `state` | `ref`/`reactive` |
| `getters` | `computed` |
| `mutations` | 直接在 action 中修改 state |
| `actions` | action 函数，可同步或异步 |
| `mapState` | `storeToRefs` |
| `modules` | 多个 `defineStore` |

Pinia 的状态修改仍然应该遵循“谁负责业务谁修改”的原则。组件可以调用 store action，不建议在模板中直接改复杂对象。

## 九、项目组织建议

```text
src/
├─ stores/
│  ├─ user.js
│  ├─ cart.js
│  └─ channel.js
├─ api/
├─ views/
└─ components/
```

把接口调用和状态变化放在对应业务 store，组件只负责展示和触发 action。跨页面数据放 store，页面专用数据留在页面组件，避免 store 变成“所有数据的垃圾桶”。

## 十、当天速记

- Pinia 是 Vue 3 推荐的状态管理库，store 由 state、getters、actions 组成。
- setup store 用 `defineStore(id, () => {})` 定义，`ref` 是状态，`computed` 是 getter，函数是 action。
- 异步请求直接写在 action 中；组件调用 action，不需要 mutations。
- `storeToRefs` 保持 state 和 getter 解构后的响应式，action 可以直接解构。
- devtools 原生支持 Pinia；持久化插件按业务需要启用。